# 1. Definição de escopo e objetivo

## 1.1. Introdução: o panorama jurídico das execuções fiscais

A cobrança de créditos públicos inscritos em dívida ativa é realizada por meio do processo de execução fiscal, conforme previsão da Lei n.º 6.830.

Porém, a demora inerente ao processo judicial somada com a utilização massificada e pouco seletiva do instituto terminou por resultar em um quadro de congestionamento do Poder Judiciário brasileiro. Nesse sentido, o estudo Justiça em Números, publicado anualmente pelo Conselho Nacional de Justiça, vem em diversas de suas edições apresentando as estatísticas que evidenciam que os processos de execução relacionam-se com esse relevante gargalo.

Com o propósito de contornar esse problema, o Supremo Tribunal Federal, ao julgar o Tema nº 1.184, estabeleceu importantes premissas para racionalizar a utilização da execução fiscal, reconhecendo a possibilidade de extinção de execuções de baixo valor e a necessidade de tentativa de cobrança ou regularização administrativa antes de ser utilizada a via judicial.

Acompanhando as conclusões do Supremo Tribunal Federal, o Conselho Nacional de Justiça editou, em 2024, a Resolução CNJ n.º 547, reproduzindo as mencionadas condicionantes em um ato normativo que orienta a atuação do Judiciário nacional.

Passados 2 anos de aplicação da normativa, mostra-se importante investigar se o propósito de redução do número de execuções fiscais foi atingido e como essa diminuição ocorreu nos diferentes órgãos do Poder Judiciário.

##1.2. Escopo do MVP e análise pretendida

Por meio do presente MVP, pretende-se investigar, de forma específica, os resultados da Resolução CNJ n.º 547 na Justiça Estadual de São Paulo.

A partir da coleta dos dados e de sua análise, objetiva-se responder os seguintes questionamentos:

1) Após a edição da Resolução CNJ n.º 547, houve redução das execuções fiscais na Justiça Estadual de São Paulo?

2) Qual o perfil de distribuição das execuções fiscais entre os órgãos do Tribunal de Justiça do Estado de São Paulo?

3) Eventual redução decorrente da aplicação da Resolução CNJ n.º 547 ocorreu de maneira uniforme entre os diversos órgãos?







# 2. METODOLOGIA E FONTE DE DADOS UTILIZADA

## 2.1. Fonte de dados: API do DATAJUD

Para a obtenção dos dados que serão utilizados para a estruturação do banco de dados e posterior extração das informações qnecessárias para as análises pretendidas, foi utilizada a API Pública do DATAJUD.

Regulamentado pela Resolução CNJ n.º 33/2020, o DATAJUD consiste na Base Nacional de Dados do Poder Judiciário, armazenando de forma centralizada as informações dos diversos órgãos que compõem o Poder Judiciário brasileiro (https://www.cnj.jus.br/sistemas/datajud/).

Além disso, a plataforma disponibiliza uma API Pública para possibilitar a consulta pública aos dados de informações processuais. As orientações para utilização do serviço constam na Wiki da plataforma (https://datajud-wiki.cnj.jus.br/api-publica).

Em síntese, por meio da utilização do método POST, o serviço retorna um arquivo JSON contendo as informações que atendam aos filtros utilizados no body da requisição.

Para a análise aqui pretendida, são necessários informações sobre: a classe do processo, uma vez que o objeto limita-se às execuções fiscais; ao tribunal, uma vez que o recorte pretendido diz respeito apenas ao Tribunal de Justiça; ao órgaõ julgador, para possibilitar a análise do padrão de distribuição entre os diversos órgãos julgadores; e a data de ajuizamento, atributo que permitirá a análise temporal da distribuição dos processos.

O glossário de dados disponibilizado constante na Wiki evidencia que essas informações estão disponíveis por meio da API, conforme se extrai dos seguintes trechos (https://datajud-wiki.cnj.jus.br/api-publica/glossario):

id	text/keyword	Identificador da origem do processo no Datajud - Chave Tribunal_Classe_Grau_OrgaoJulgador_NumeroProcesso
tribunal	text/keyword	Identificação do Tribunal pela sigla
numeroProcesso	text/keyword	Numeração Única (CNJ) do processo sem formatação
dataAjuizamento	datetime	Data de ajuizamento da capa do processo
classe	object{}	Classe Processual conforme TPU
classe.codigo	long	Código da classe processual
classe.nome	text/keyword	Descrição da classe processual


## 2.2. Publicidade dos dados e limites de uso

Os dados disponibilizados por meio da API de Consulta Pública do DATAJUD são públicos e equivalem apenas aos metadados processuais, não englobando informações abrangidas por segredo de justiça ou funcional, tampouco dados pessoais protegidos pela Lei Geral de Proteção de Dados.

Por sua vez, os termos de uso da API constam no termo de uso encontrado na Wiki e que também foi juntada à pasta deste projeto.

Entre as condições de maior importância para as atividades deste trabalho, está a limitação de 120 requisições por minuto. Para atendimento desta condição, na etapa de ingestão dos dados, foram inseridas condições para garantir a observância do limite e o cumprimento às condições de uso da plataforma, em especial a inclusão do comando  **time.sleep(1.2)** na execução das chamadas da API.


# 3. INGESTÃO DE DADOS


Para a ingestão dos dados da API Pública do DATAJUD, foi necessário considerar algumas condições operacionais da própria ferramenta de consulta.

Primeiramente, a API possui limite de até 10.000 registros por consulta. Para possibilitar a recuperação de um volume superior de dados, tornou-se necessária a utilização de paginação por meio do parâmetro search_after, que permite dar continuidade à consulta a partir do último registro retornado na página anterior. Essa necessidade exigiu a inserção, no código, de variáveis para controlar a paginação e a continuidade da lista:

> **Trechos do código da célula 6**

> (...)

>      pagina += 1

> (...)

>         ultimo_sort = resultados[-1]["sort"]

>         consulta["search_after"] = ultimo_sort



Por outro lado, para evitar a execução de consultas excessivamente longas e reduzir o impacto de eventuais falhas durante o processo de extração, optou-se por realizar as pesquisas em períodos mensais. Dessa forma, cada mês passou a representar uma unidade independente de ingestão, permitindo que os dados fossem extraídos, validados e persistidos separadamente.

Assim, considerando o período objeto da análise, compreendido entre janeiro de 2023 e junho de 2026, foi adotada uma metodologia baseada em uma tabela de controle de ingestão, contendo, para cada período mensal, a data inicial, a data final, o status da execução, a quantidade de registros ingeridos, a data da ingestão e eventual mensagem de erro. 

A criação dos períodos foi feita diretamente na tabela ControleIngestao por meio dos comandos EXPLODE, SEQUENCE e INTERVAL, o que possibilitou a criação dos campos correspondentes aos meses compreendidos no período de análise:

> FROM (SELECT EXPLODE(SEQUENCE(DATE '2023-01-01', DATE '2026-06-01', INTERVAL 1 MONTH))AS mes

Uma outra necessidade técnica para consistência dos dados foi observada durante os testes. Durante a execução das consultas, percebeu-se que era possível que alguma página não era retornada corretamente em virtude de algum erro, como por exemplo os erros 429 (Too Many Requests) e 504 (Gateway Timeout).

A ausência de identificação da ocorrência de erro em uma das páginas poderia conduzir a problemas graves de consistência na análise, uma vez que os dados persistidos seriam apenas parciais e não retratariam a totalidade dos processos daquele período.

Por esse motivo, mostrou-se necessária a criação de um campo de controle do sucesso do retorno da consulta, o campo "Status". A partir desse campo, a execução da função configurada para fazer a consulta identificava se ocorreu erro persistente no retorno e, caso tenha ocorrido, não prosseguia com a criação do dataframe e persistência das informaçõe na camada Bronze.

Em outras palavras, a persistência dos dados somente ocorreria após a conclusão integral da extração correspondente ao período mensal. Caso a execução fosse interrompida ou permanecesse sem sucesso após o número máximo de tentativas, nenhum dado parcial daquele período seria gravado na camada Bronze. Nessa situação, a tabela de controle seria atualizada com o status de erro, permitindo a retomada posterior da ingestão.

Além disso, também se entendeu pertinente criar, por meio de uma estrutura de iteração, a possibilidade de reexecução da consulta na hipótese de erro até o limite de 3 tentativas. Essa construção viabilizava aproveitar a execução da consulta em casos em que o erro era transitório e poderia já não ser exibido na outra tentativa.









In [0]:
%sql
CREATE OR REPLACE TABLE ControleIngestao
USING DELTA
AS

SELECT
    DATE_FORMAT(mes, 'yyyy-MM') AS periodo,
    mes AS data_inicio,
    ADD_MONTHS(mes, 1) AS data_fim,
    'PENDENTE' AS status,
    CAST(NULL AS BIGINT) AS quantidade_registros,
    CAST(NULL AS TIMESTAMP) AS data_ingestao
FROM (
    SELECT
        EXPLODE(
            SEQUENCE(
                DATE '2023-01-01',
                DATE '2026-06-01',
                INTERVAL 1 MONTH
            )
        ) AS mes
);

In [0]:
import requests
import time
from pyspark.sql import functions as fsql


#Variáveis utilizadas na chamada da API

url='https://api-publica.datajud.cnj.jus.br/api_publica_tjsp/_search'

headers = {
    "Authorization": "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw==",
    "Content-Type": "application/json"
}


# Função de ingestão pela API
def extrair_processos(data_inicio, data_fim):

    consulta = {
        "size": 5000,
        "query": {
            "bool": {
                "filter": [
                    {
                        "term": {
                            "classe.codigo": 1116
                        }
                    },
                    {
                        "range": {
                            "dataAjuizamento": {
                                "gte": data_inicio,
                                "lt": data_fim
                            }
                        }
                    }
                ]
            }
        },
        "sort": [
            {"dataAjuizamento": "asc"}
        ]
    }

    Processos = []
    pagina = 1

    while True:

        tentativa = 1
        max_tentativas = 3

        while tentativa <= max_tentativas:

            response = requests.post(
                url,
                headers=headers,
                json=consulta
            )

            if response.status_code == 200:
                break

            if response.status_code in [429, 504]:

                print(
                    f"HTTP {response.status_code} - "
                    f"tentativa {tentativa}/{max_tentativas}"
                )

                time.sleep(10 * tentativa)
                tentativa += 1

            else:

                return (
                    False,
                    [],
                    f"Erro HTTP {response.status_code}"
                )

        # chegou ao limite de tentativas
        if response.status_code != 200:

            return (
                False,
                [],
                f"Erro HTTP {response.status_code} após {max_tentativas} tentativas"
            )

        dados = response.json()
        resultados = dados["hits"]["hits"]

        if len(resultados) == 0:
            break

        for processo in resultados:

            novo_processo = processo["_source"]

            novo_registro = {
                "numero_processo": novo_processo["numeroProcesso"],
                "tribunal": novo_processo["tribunal"],
                "data_ajuizamento": novo_processo["dataAjuizamento"],
                "grau": novo_processo["grau"],
                "classe_codigo": novo_processo["classe"]["codigo"],
                "classe_nome": novo_processo["classe"]["nome"],
                "orgaoJulgador_nome":
                    novo_processo["orgaoJulgador"]["nome"]
            }

            Processos.append(novo_registro)

        print(
            f"Página {pagina}: "
            f"{len(resultados)} registros"
        )

        pagina += 1

        ultimo_sort = resultados[-1]["sort"]
        consulta["search_after"] = ultimo_sort

        time.sleep(1.2)

    return True, Processos, None


# Ingestão de dados por período

periodos_pendentes = (
    spark.table("ControleIngestao")
    .filter(
        fsql.col("status").isin("PENDENTE", "ERRO")
    )
    .orderBy("data_inicio")
)

for linha in periodos_pendentes.collect():

    periodo = linha["periodo"]

    data_inicio = linha["data_inicio"].strftime("%Y%m%d%H%M%S")
    data_fim = linha["data_fim"].strftime("%Y%m%d%H%M%S")

    print(f"Iniciando período {periodo}")

    # Marca o período como PROCESSANDO
    spark.sql(f"""
        UPDATE ControleIngestao
        SET
            status = 'PROCESSANDO',
            mensagem_erro = NULL
        WHERE periodo = '{periodo}'
    """)

    sucesso, processos_mes, erro = extrair_processos(
        data_inicio,
        data_fim
    )

    if sucesso:

        if len(processos_mes) > 0:

            df_mes = spark.createDataFrame(processos_mes)

            df_mes = (
                df_mes
                .withColumn(
                    "periodo_ingestao",
                    fsql.lit(periodo)
                )
                .withColumn(
                    "data_ingestao",
                    fsql.current_timestamp()
                )
            )

            df_mes.write \
                .mode("append") \
                .saveAsTable("Bronze_Processos")

        # Atualiza controle após conclusão
        spark.sql(f"""
            UPDATE ControleIngestao
            SET
                status = 'OK',
                quantidade_registros = {len(processos_mes)},
                data_ingestao = CURRENT_TIMESTAMP(),
                mensagem_erro = NULL
            WHERE periodo = '{periodo}'
        """)

        print(
            f"{periodo} concluído: "
            f"{len(processos_mes)} registros."
        )

    else:

        spark.sql(f"""
            UPDATE ControleIngestao
            SET
                status = 'ERRO',
                quantidade_registros = NULL,
                mensagem_erro = '{erro}'
            WHERE periodo = '{periodo}'
        """)

        print(
            f"Falha na ingestão de {periodo}: {erro}"
        )
